In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from collections import defaultdict

In [3]:
def preprocess(df):
    # '회 이상' 제거 후 정수 변환
    count_cols_회 = ['캠페인접촉건수_R12M']
    for col in count_cols_회:
        df[col] = df[col].astype(str).str.replace('회 이상', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(int)

    # '일 이상' 제거 후 정수 변환
    count_cols_일 = ['캠페인접촉일수_R12M']
    for col in count_cols_일:
        df[col] = df[col].astype(str).str.replace('일 이상', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(int)

    return df


df1 = preprocess(pd.read_parquet('train/7.마케팅정보/201807_train_마케팅정보.parquet'))
df2 = preprocess(pd.read_parquet('train/7.마케팅정보/201808_train_마케팅정보.parquet'))
df3 = preprocess(pd.read_parquet('train/7.마케팅정보/201809_train_마케팅정보.parquet'))
df4 = preprocess(pd.read_parquet('train/7.마케팅정보/201810_train_마케팅정보.parquet'))
df5 = preprocess(pd.read_parquet('train/7.마케팅정보/201811_train_마케팅정보.parquet'))
df6 = preprocess(pd.read_parquet('train/7.마케팅정보/201812_train_마케팅정보.parquet'))

In [4]:
dfs = [df.drop(columns=['기준년월'], errors='ignore') for df in [df1, df2, df3, df4, df5, df6]]

def merge_two_avg(df_left, df_right):
    merge_keys = ['ID']
    if 'Segment' in df_left.columns and 'Segment' in df_right.columns:
        merge_keys.append('Segment')

    merged = pd.merge(df_left, df_right, on=merge_keys, how='outer', suffixes=('_left', '_right'))
    result = merged[merge_keys].copy()
    
    # 평균 계산
    for col in set(df_left.columns).union(df_right.columns):
        if col in merge_keys:
            continue
        col_left = f"{col}_left" if f"{col}_left" in merged.columns else None
        col_right = f"{col}_right" if f"{col}_right" in merged.columns else None
        
        cols_to_avg = [c for c in [col_left, col_right] if c is not None]
        result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
    
    return result

from functools import reduce
merged_df = reduce(merge_two_avg, dfs)
merged_df

,ID,컨택건수_리볼빙_인터넷_B0M,컨택건수_이용유도_TM_R6M,컨택건수_리볼빙_인터넷_R6M,컨택건수_리볼빙_LMS_B0M,컨택건수_CA_LMS_B0M,컨택건수_카드론_EM_B0M,컨택건수_이용유도_인터넷_B0M,컨택건수_보험_TM_R6M,컨택건수_이용유도_인터넷_R6M,...,컨택건수_이용유도_LMS_B0M,컨택건수_CA_EM_R6M,컨택건수_리볼빙_청구서_R6M,컨택건수_리볼빙_TM_B0M,컨택건수_부대서비스_TM_B0M,컨택건수_이용유도_LMS_R6M,컨택건수_CA_청구서_B0M,컨택건수_리볼빙_LMS_R6M,캠페인접촉일수_R12M,컨택건수_부대서비스_TM_R6M
0,TRAIN_000000,0.0,1.50000,0.0,0.0,0.0,0.0,0.00000,1.50,0.53125,...,2.0000,0.0,0.0,0.0,0.0,9.46875,0.0,0.0,1.00000,0.0
1,TRAIN_000001,0.0,1.46875,0.0,0.0,0.0,0.0,0.34375,0.00,4.34375,...,1.0625,0.0,0.0,0.0,0.0,4.31250,0.0,0.0,14.84375,0.0
2,TRAIN_000002,0.0,0.87500,0.0,0.0,0.0,0.0,0.00000,0.00,1.37500,...,0.0000,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,1.00000,0.0
3,TRAIN_000003,0.0,1.40625,0.0,0.0,0.0,0.0,0.00000,0.75,0.00000,...,2.0000,0.0,0.0,0.0,0.0,10.78125,0.0,0.0,1.50000,0.0
4,TRAIN_000004,0.0,5.00000,0.0,0.0,0.0,0.0,0.00000,0.00,0.59375,...,2.0625,0.0,0.0,0.0,0.0,14.03125,0.0,0.0,1.00000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.0,2.18750,0.0,0.0,0.0,0.0,0.00000,0.00,1.34375,...,0.6250,0.0,0.0,0.0,0.0,3.93750,0.0,0.0,1.00000,0.0
399996,TRAIN_399996,0.0,1.25000,0.0,0.0,0.0,0.0,0.00000,0.00,0.00000,...,2.0000,0.0,0.0,0.0,0.0,10.12500,0.0,0.0,15.62500,0.0
399997,TRAIN_399997,0.0,0.00000,0.0,0.0,0.0,0.0,0.00000,0.00,0.56250,...,0.0000,0.0,0.0,0.0,0.0,0.59375,0.0,0.0,1.00000,0.0
399998,TRAIN_399998,0.0,0.00000,0.0,0.0,0.0,0.0,0.03125,0.00,1.96875,...,0.0000,0.0,0.0,0.0,0.0,0.75000,0.0,0.0,1.00000,0.0


In [5]:
df_segment = pd.read_parquet('train/1.회원정보/201807_train_회원정보.parquet')[['ID', 'Segment']]

# 2. 중복 제거 (ID별 Segment가 유일하다는 전제)
df_segment = df_segment.drop_duplicates(subset='ID')

# 3. merged_df에 Segment 열 붙이기 (ID 기준)
merged_df = pd.merge(merged_df, df_segment, on='ID', how='left')
merged_df

,ID,컨택건수_리볼빙_인터넷_B0M,컨택건수_이용유도_TM_R6M,컨택건수_리볼빙_인터넷_R6M,컨택건수_리볼빙_LMS_B0M,컨택건수_CA_LMS_B0M,컨택건수_카드론_EM_B0M,컨택건수_이용유도_인터넷_B0M,컨택건수_보험_TM_R6M,컨택건수_이용유도_인터넷_R6M,...,컨택건수_CA_EM_R6M,컨택건수_리볼빙_청구서_R6M,컨택건수_리볼빙_TM_B0M,컨택건수_부대서비스_TM_B0M,컨택건수_이용유도_LMS_R6M,컨택건수_CA_청구서_B0M,컨택건수_리볼빙_LMS_R6M,캠페인접촉일수_R12M,컨택건수_부대서비스_TM_R6M,Segment
0,TRAIN_000000,0.0,1.50000,0.0,0.0,0.0,0.0,0.00000,1.50,0.53125,...,0.0,0.0,0.0,0.0,9.46875,0.0,0.0,1.00000,0.0,D
1,TRAIN_000001,0.0,1.46875,0.0,0.0,0.0,0.0,0.34375,0.00,4.34375,...,0.0,0.0,0.0,0.0,4.31250,0.0,0.0,14.84375,0.0,E
2,TRAIN_000002,0.0,0.87500,0.0,0.0,0.0,0.0,0.00000,0.00,1.37500,...,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,1.00000,0.0,C
3,TRAIN_000003,0.0,1.40625,0.0,0.0,0.0,0.0,0.00000,0.75,0.00000,...,0.0,0.0,0.0,0.0,10.78125,0.0,0.0,1.50000,0.0,D
4,TRAIN_000004,0.0,5.00000,0.0,0.0,0.0,0.0,0.00000,0.00,0.59375,...,0.0,0.0,0.0,0.0,14.03125,0.0,0.0,1.00000,0.0,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.0,2.18750,0.0,0.0,0.0,0.0,0.00000,0.00,1.34375,...,0.0,0.0,0.0,0.0,3.93750,0.0,0.0,1.00000,0.0,E
399996,TRAIN_399996,0.0,1.25000,0.0,0.0,0.0,0.0,0.00000,0.00,0.00000,...,0.0,0.0,0.0,0.0,10.12500,0.0,0.0,15.62500,0.0,D
399997,TRAIN_399997,0.0,0.00000,0.0,0.0,0.0,0.0,0.00000,0.00,0.56250,...,0.0,0.0,0.0,0.0,0.59375,0.0,0.0,1.00000,0.0,C
399998,TRAIN_399998,0.0,0.00000,0.0,0.0,0.0,0.0,0.03125,0.00,1.96875,...,0.0,0.0,0.0,0.0,0.75000,0.0,0.0,1.00000,0.0,E


In [6]:
nan_columns = merged_df.columns[merged_df.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]


In [7]:
ex1 = merged_df

In [8]:
missing_mask = ex1.isna() | (ex1 == -1)
missing_ratio = missing_mask.mean()

high_na = missing_ratio[missing_ratio > 0.2].index.tolist()

high_const_cols = []
threshold_const = 0.8

for col in ex1.columns:
    top_ratio = ex1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_ratio > threshold_const:
        high_const_cols.append(col)

to_drop = list(set(high_na + high_const_cols))

if 'Segment' in to_drop:
    to_drop.remove('Segment')

print("삭제 대상 컬럼 (결측>20% 또는 동일값>80%):", to_drop)

ex1.drop(columns=to_drop, inplace=True)

삭제 대상 컬럼 (결측>20% 또는 동일값>80%): ['컨택건수_리볼빙_인터넷_B0M', '컨택건수_리볼빙_인터넷_R6M', '컨택건수_리볼빙_LMS_B0M', '컨택건수_카드론_EM_B0M', '컨택건수_CA_LMS_B0M', '컨택건수_CA_LMS_R6M', '컨택건수_CA_당사앱_B0M', '컨택건수_리볼빙_청구서_B0M', '컨택건수_이용유도_당사앱_R6M', '컨택건수_카드론_청구서_R6M', '컨택건수_카드론_인터넷_R6M', '컨택건수_CA_인터넷_R6M', '컨택건수_카드론_당사앱_B0M', '컨택건수_이용유도_TM_B0M', '컨택건수_이용유도_당사앱_B0M', '컨택건수_포인트소진_TM_B0M', '컨택건수_채권_R6M', '컨택건수_리볼빙_당사앱_B0M', '컨택건수_리볼빙_TM_R6M', '컨택건수_채권_B0M', '컨택건수_CA_EM_B0M', '컨택건수_카드론_인터넷_B0M', '컨택건수_포인트소진_TM_R6M', '컨택건수_카드론_LMS_B0M', '컨택건수_리볼빙_EM_R6M', '컨택건수_FDS_B0M', '컨택건수_카드론_청구서_B0M', '컨택건수_CA_인터넷_B0M', '컨택건수_신용발급_TM_B0M', '컨택건수_리볼빙_당사앱_R6M', '컨택건수_CA_TM_R6M', '컨택건수_카드론_EM_R6M', '컨택건수_카드론_LMS_R6M', '컨택건수_CA_TM_B0M', '컨택건수_리볼빙_EM_B0M', '컨택건수_CA_당사앱_R6M', '컨택건수_카드론_TM_B0M', '컨택건수_CA_청구서_R6M', '컨택건수_카드론_당사앱_R6M', '컨택건수_FDS_R6M', '컨택건수_신용발급_TM_R6M', '컨택건수_보험_TM_B0M', '컨택건수_CA_EM_R6M', '컨택건수_리볼빙_청구서_R6M', '컨택건수_리볼빙_TM_B0M', '컨택건수_부대서비스_TM_B0M', '컨택건수_CA_청구서_B0M', '컨택건수_리볼빙_LMS_R6M', '컨택건수_부대서비스_TM_R6M']


In [9]:
num_df = ex1.select_dtypes(include=[np.number]).dropna()

corr = num_df.corr().abs()

high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.8]

if not np.issubdtype(ex1['Segment'].dtype, np.number):
    segment_map = {label: idx for idx, label in enumerate(sorted(ex1['Segment'].unique()))}
    ex1['Segment_encoded'] = ex1['Segment'].map(segment_map)
else:
    ex1['Segment_encoded'] = ex1['Segment']

segment_corr = ex1[num_df.columns].corrwith(ex1['Segment_encoded']).abs()

high_corr_pairs['Corr_with_Segment_1'] = high_corr_pairs['Feature_1'].map(segment_corr)
high_corr_pairs['Corr_with_Segment_2'] = high_corr_pairs['Feature_2'].map(segment_corr)

high_corr_pairs = high_corr_pairs.sort_values(by='Correlation', ascending=False).reset_index(drop=True)

print(f"▶ 상관계수 0.7 초과 변수쌍 수: {len(high_corr_pairs)}")
display(high_corr_pairs)


▶ 상관계수 0.7 초과 변수쌍 수: 7


,Feature_1,Feature_2,Correlation,Corr_with_Segment_1,Corr_with_Segment_2
0,캠페인접촉건수_R12M,캠페인접촉일수_R12M,0.988083,0.020296,0.021868
1,컨택건수_이용유도_EM_B0M,컨택건수_이용유도_EM_R6M,0.974853,0.133799,0.134332
2,컨택건수_이용유도_LMS_B0M,컨택건수_이용유도_LMS_R6M,0.942083,0.062155,0.034293
3,컨택건수_카드론_TM_R6M,캠페인접촉일수_R12M,0.896994,0.021377,0.021868
4,컨택건수_카드론_TM_R6M,캠페인접촉건수_R12M,0.896114,0.021377,0.020296
5,컨택건수_이용유도_청구서_B0M,컨택건수_이용유도_청구서_R6M,0.893161,0.138899,0.073554
6,컨택건수_이용유도_인터넷_B0M,컨택건수_이용유도_인터넷_R6M,0.840517,0.010529,0.049057


In [10]:
to_drop = []

for _, row in high_corr_pairs.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    c1, c2 = row['Corr_with_Segment_1'], row['Corr_with_Segment_2']
    
    if pd.isna(c1) or pd.isna(c2):
        continue
    
    if c1 < c2:
        to_drop.append(f1)
    else:
        to_drop.append(f2)

to_drop = list(set(to_drop))

# 결과 출력
print(f"▶ 제거 대상 피처 수: {len(to_drop)}")
print("제거할 피처 목록:")
print(to_drop)

▶ 제거 대상 피처 수: 6
제거할 피처 목록:
['컨택건수_카드론_TM_R6M', '컨택건수_이용유도_인터넷_B0M', '컨택건수_이용유도_청구서_R6M', '컨택건수_이용유도_LMS_R6M', '캠페인접촉건수_R12M', '컨택건수_이용유도_EM_B0M']


In [11]:
cols_to_drop = ['컨택건수_카드론_TM_R6M', '컨택건수_이용유도_인터넷_B0M', '컨택건수_이용유도_청구서_R6M', '컨택건수_이용유도_LMS_R6M', '캠페인접촉건수_R12M', '컨택건수_이용유도_EM_B0M']
ex1.drop(columns=cols_to_drop, inplace=True)

In [12]:
cols_to_drop = ['Segment_encoded']
ex1.drop(columns=cols_to_drop, inplace=True)
cols = ex1.columns.tolist()
cols

['ID',
 '컨택건수_이용유도_TM_R6M',
 '컨택건수_보험_TM_R6M',
 '컨택건수_이용유도_인터넷_R6M',
 '컨택건수_이용유도_EM_R6M',
 '컨택건수_이용유도_청구서_B0M',
 '컨택건수_이용유도_LMS_B0M',
 '캠페인접촉일수_R12M',
 'Segment']

In [14]:
ex1.to_parquet('마케팅_전처리_Segment.parquet', index=False)

In [15]:
ddf1 = preprocess(pd.read_parquet('train/7.마케팅정보/201807_train_마케팅정보.parquet'))
ddf2 = preprocess(pd.read_parquet('train/7.마케팅정보/201808_train_마케팅정보.parquet'))
ddf3 = preprocess(pd.read_parquet('train/7.마케팅정보/201809_train_마케팅정보.parquet'))
ddf4 = preprocess(pd.read_parquet('train/7.마케팅정보/201810_train_마케팅정보.parquet'))
ddf5 = preprocess(pd.read_parquet('train/7.마케팅정보/201811_train_마케팅정보.parquet'))
ddf6 = preprocess(pd.read_parquet('train/7.마케팅정보/201812_train_마케팅정보.parquet'))

In [16]:

dfs = [ddf1, ddf2, ddf3, ddf4, ddf5, ddf6]
for i in range(len(dfs)):
    if '기준년월' in dfs[i].columns:
        dfs[i] = dfs[i].drop(columns=['기준년월'])

# ID 기준으로 병합 후 평균
from functools import reduce

merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='ID', how='outer', suffixes=('', '_dup')),
    dfs
)

# 같은 이름의 열 평균 구하기
from collections import defaultdict
import pandas as pd

result = pd.DataFrame()
result['ID'] = merged_df['ID']

# 열 이름 모아 평균 구하기
col_dict = defaultdict(list)
for col in merged_df.columns:
    if col != 'ID':
        base_col = col.split('_dup')[0]
        col_dict[base_col].append(col)

for base_col, cols in col_dict.items():
    result[base_col] = merged_df[cols].mean(axis=1, skipna=True)

# 결과 확인
print(result.head())

             ID  컨택건수_카드론_TM_B0M  컨택건수_리볼빙_TM_B0M  컨택건수_CA_TM_B0M  \
0  TRAIN_000000         0.000000              0.0             0.0   
1  TRAIN_000001         0.115385              0.0             0.0   
2  TRAIN_000002         0.000000              0.0             0.0   
3  TRAIN_000003         0.000000              0.0             0.0   
4  TRAIN_000004         0.000000              0.0             0.0   

   컨택건수_이용유도_TM_B0M  컨택건수_신용발급_TM_B0M  컨택건수_부대서비스_TM_B0M  컨택건수_포인트소진_TM_B0M  \
0          0.000000               0.0                0.0                0.0   
1          0.000000               0.0                0.0                0.0   
2          0.000000               0.0                0.0                0.0   
3          0.000000               0.0                0.0                0.0   
4          0.576923               0.0                0.0                0.0   

   컨택건수_보험_TM_B0M  컨택건수_카드론_LMS_B0M  ...  컨택건수_카드론_당사앱_R6M  컨택건수_CA_당사앱_R6M  \
0             0.0              

In [17]:
cols = ['ID',
 '컨택건수_이용유도_TM_R6M',
 '컨택건수_보험_TM_R6M',
 '컨택건수_이용유도_인터넷_R6M',
 '컨택건수_이용유도_EM_R6M',
 '컨택건수_이용유도_청구서_B0M',
 '컨택건수_이용유도_LMS_B0M',
 '캠페인접촉일수_R12M',]

result = result[cols]
result

,ID,컨택건수_이용유도_TM_R6M,컨택건수_보험_TM_R6M,컨택건수_이용유도_인터넷_R6M,컨택건수_이용유도_EM_R6M,컨택건수_이용유도_청구서_B0M,컨택건수_이용유도_LMS_B0M,캠페인접촉일수_R12M
0,TRAIN_000000,2.423077,2.423077,0.846154,62.961538,0.000000,2.000000,1.000000
1,TRAIN_000001,2.384615,0.000000,5.038462,0.653846,0.000000,0.769231,14.807692
2,TRAIN_000002,1.423077,0.000000,2.038462,16.038462,0.000000,0.000000,1.000000
3,TRAIN_000003,2.192308,0.576923,0.000000,29.038462,0.038462,2.000000,1.769231
4,TRAIN_000004,6.230769,0.000000,1.192308,1.730769,0.423077,2.230769,1.000000
...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,2.884615,0.000000,2.500000,1.423077,0.000000,0.769231,1.000000
399996,TRAIN_399996,1.807692,0.000000,0.000000,52.884615,0.000000,2.000000,15.961538
399997,TRAIN_399997,0.000000,0.000000,1.038462,0.576923,1.000000,0.000000,1.000000
399998,TRAIN_399998,0.000000,0.000000,2.576923,1.076923,0.615385,0.000000,1.000000


In [18]:
result.to_parquet('마케팅_전처리_test.parquet', index=False)

In [19]:
nan_columns = result.columns[result.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]
